# Résultats complets — BUSI + CMMD + données cliniques réelles

Notebook unique produisant **tous les résultats de la soutenance**.

| Étape | Ce qu'elle produit |
|---|---|
| Baselines | Échographie seule, mammographie seule, **clinique seule** |
| Fusion | Moyenne des logits + cross-attention AMW sur paires virtuelles |
| DeLong + Holm | Comparaisons appariées, p-values corrigées |
| Grad-CAM + IoU | Le modèle regarde-t-il la lésion ? (mesuré, pas illustré) |
| SHAP | Part des variables cliniques dans la décision |

**Données** : BUSI (647 échographies), CMMD (3 744 mammographies, **âge réel de 1 775 patientes**), variables cliniques C0 réelles uniquement.

> La baseline échographie est évaluée sur BUSI, tandis que mammographie et fusion sont évaluées sur CMMD. Ces lignes ne sont donc pas une comparaison appariée entre modalités ; les comparaisons fusion/mammographie et fusion/clinique utilisent les mêmes patientes CMMD. Les paires fusion sont virtuelles, pas des examens de la même patiente.

**Plan détaillé : 29 expériences = 28 entraînements et 1 contrôle d'intégrité sans réentraînement.**

| # | Expérience | # | Expérience |
|---:|---|---:|---|
| 1 | `echo_seule` — BUSI | 2 | `mammo_seule` — CMMD |
| 3 | `clinique_seule` — CMMD, âge + type de lésion | 4 | `logit_averaging_pair0` |
| 5 | `logit_averaging_pair1` | 6 | `logit_averaging_pair2` |
| 7 | `logit_averaging_pair3` | 8 | `logit_averaging_pair4` |
| 9 | `early_pair0` | 10 | `early_pair1` |
| 11 | `early_pair2` | 12 | `early_pair3` |
| 13 | `early_pair4` | 14 | `late_pair0` |
| 15 | `late_pair1` | 16 | `late_pair2` |
| 17 | `late_pair3` | 18 | `late_pair4` |
| 19 | `cross_attention_pair0` | 20 | `cross_attention_pair1` |
| 21 | `cross_attention_pair2` | 22 | `cross_attention_pair3` |
| 23 | `cross_attention_pair4` | 24 | `cross_attention_amw_pair0` |
| 25 | `cross_attention_amw_pair1` | 26 | `cross_attention_amw_pair2` |
| 27 | `cross_attention_amw_pair3` | 28 | `cross_attention_amw_pair4` |
| 29 | `integrity_shuffle` — hold-out permuté, modèles inchangés | | |

**Ressources** : GPU CUDA recommandé ≥16 Gio (T4, fusion batch 16, AMP) ; viser ≥16 Gio de RAM système et ≥20 Gio libres sur Drive pour cache, checkpoints et résultats. Pour un GPU plus petit, régler `FUSION_BATCH_SIZE` (et `BASELINE_IMAGE_BATCH_SIZE`) à 8 avant le lancement ; la configuration est intégrée aux fingerprints. Le temps par expérience est inconnu : chronométrer les baselines avant d'estimer le total. Priorité : baselines → `logit_averaging`, `late`, `cross_attention`, `cross_attention_amw` (cinq tirages chacun) → `early` (complément). Si le délai empêche la dernière stratégie, la déclarer non exécutée plutôt que réduire les tirages ou présenter une comparaison incomplète comme exhaustive.

> ⚠️ Exécutez les cellules **dans l'ordre**. Les checkpoints reprennent au dernier epoch sauvegardé ; les résultats complets déjà exportés sont rechargés et non réentraînés.
>
> ⚠️ L'appariement label-conditionné des anciennes versions révélait le label en validation/test. La version actuelle tire les paires indépendamment des labels dans chaque split. Les modalités proviennent de personnes différentes : la fusion est un protocole exploratoire de paires virtuelles, pas une évaluation clinique multimodale appariée.
>
> ⚠️ BUSI ne publie pas la correspondance image-patiente. Le manifeste attribue un identifiant par fichier ; le découpage BUSI par ces identifiants ne garantit donc pas un hold-out indépendant par patiente. La baseline BUSI doit rester exploratoire et ne peut pas être présentée comme une validation patient-indépendante.

In [ ]:
# ── 1. Installation, Drive, dépôt ────────────────────────────────────────
# pydicom est INDISPENSABLE : CMMD est stocké en DICOM et Colab ne l'a pas.
%pip -q install timm albumentations shap pydicom --no-warn-script-location
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, os, zipfile, time, json, hashlib, gc
import numpy as np, pandas as pd

DRIVE = Path('/content/drive/MyDrive')
zips = sorted(DRIVE.glob('projet*.zip')) or sorted(DRIVE.glob('*breast*multimodal*.zip'))
assert zips, f'Aucun zip du projet dans {DRIVE}'
REPO = Path('/content/breast-multimodal-fusion')
if not REPO.exists():
    t0 = time.time()
    with zipfile.ZipFile(zips[0]) as z: z.extractall('/content/_unzip')
    c = list(Path('/content/_unzip').rglob('data/manifest.csv'))
    assert c, 'manifest.csv absent du zip'
    c[0].parents[1].rename(REPO)
    print(f'décompressé en {time.time()-t0:.0f}s')
os.chdir(REPO); sys.path.insert(0, str(REPO))

OUT = DRIVE / 'resultats_soutenance'; OUT.mkdir(exist_ok=True)
CKPT = DRIVE / 'breast-multimodal-checkpoints'; CKPT.mkdir(exist_ok=True)
RUN_DIR = OUT / 'runs'; RUN_DIR.mkdir(exist_ok=True)
FIG_DIR = OUT / 'figures'; FIG_DIR.mkdir(exist_ok=True)

import torch
assert torch.cuda.is_available(), "Exécution → Modifier le type d'exécution → GPU"
print('GPU :', torch.cuda.get_device_name(0))
print(f'Mémoire GPU totale : {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} Gio')
try:
    import psutil
    print(f'Mémoire système disponible : {psutil.virtual_memory().available / 2**30:.1f} Gio')
except ImportError:
    print('psutil indisponible : mémoire système non mesurée')


In [ ]:
# ── 2. Garde-fous méthodologiques ────────────────────────────────────────
# Deux minutes de vérification avant une heure de calcul.
!python -m pytest tests/test_no_patient_leakage.py tests/test_localization.py tests/test_pairing.py tests/test_trainer_resume.py -q --no-header

In [ ]:
# ── 3. Cache images 256 px (étape la plus rentable du notebook) ──────────
# Les DICOM de CMMD se lisent ~5x plus lentement qu'un PNG, et les décoder à
# chaque époque gâcherait l'essentiel du temps GPU. On convertit une fois vers
# le disque local, puis on entraîne dessus.
from concurrent.futures import ThreadPoolExecutor
import cv2
from src.data.dicom_convert import dicom_to_array

CACHE = Path('/content/cache256'); CACHE.mkdir(exist_ok=True)
CACHE_ARCHIVE = DRIVE / 'cache256.zip'
if not any(CACHE.iterdir()) and CACHE_ARCHIVE.exists():
    with zipfile.ZipFile(CACHE_ARCHIVE) as archive:
        archive.extractall(CACHE)
    print('Cache restauré depuis Drive')
ERRORS = []   # les échecs silencieux coûtent des heures : on les remonte
SIZE, ROOT = 256, REPO / 'data/raw'
manifest = pd.read_csv('data/manifest.csv')
manifest['image_path'] = manifest['image_path'].astype(str).str.replace(chr(92), '/', regex=False)
manifest['mask_path'] = manifest['mask_path'].fillna('').astype(str).str.replace(chr(92), '/', regex=False)

def cache_one(rel: str):
    rel = str(rel).replace(chr(92), '/')
    dst = CACHE / (rel.replace('/', '__').rsplit('.', 1)[0] + '.png')
    if dst.exists() and cv2.imread(str(dst), cv2.IMREAD_UNCHANGED) is not None:
        return str(dst)
    try:
        src = ROOT / rel
        img = dicom_to_array(src) if src.suffix.lower() == '.dcm' \
              else cv2.imread(str(src), cv2.IMREAD_UNCHANGED)
        if img is None: raise FileNotFoundError(f'Image source illisible : {src}')
        if img.dtype == np.uint16: img = (img / 257).astype(np.uint8)
        if img.ndim == 3: img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        if not cv2.imwrite(str(dst), cv2.resize(img, (SIZE, SIZE), interpolation=cv2.INTER_AREA)):
            raise OSError(f'Écriture du cache impossible : {dst}')
        return str(dst)
    except Exception as exc:
        ERRORS.append(f'{rel} : {type(exc).__name__} — {exc}')
        return None

work = manifest[manifest.dataset.isin(['busi', 'cmmd'])].copy()
print(f'{len(work)} images à mettre en cache (BUSI + CMMD)...')
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    work['cached'] = list(ex.map(cache_one, work.image_path))

# Masques BUSI, pour la validation Grad-CAM
msk = work[(work.dataset == 'busi') & work.mask_path.notna() & work.mask_path.astype(str).str.len().gt(0)]
with ThreadPoolExecutor(max_workers=16) as ex:
    cached_masks = list(ex.map(cache_one, msk.mask_path))
work.loc[msk.index, 'cached_mask'] = cached_masks

if ERRORS:
    details = '\n'.join(ERRORS[:10])
    raise RuntimeError(f'{len(ERRORS)} échec(s) pendant la mise en cache. Premiers détails :\n{details}')

work = work[work.cached.notna()].reset_index(drop=True)
print(f'{len(work)} images en cache · {time.time()-t0:.0f}s')
print(work.groupby('dataset').agg(n=('label','size'), malignes=('label','mean')).round(3))

# Contrôle : les DEUX modalités doivent être présentes, sinon la fusion
# n'a plus de sens et les cellules suivantes échoueraient sans prévenir.
manquants = {'busi', 'cmmd'} - set(work.dataset)
assert not manquants, f'Modalites absentes du cache : {manquants} — pydicom installe (cellule 1) ?'
assert len(work) == len(manifest), f'Cache incomplet : {len(work)} / {len(manifest)} images'
if not CACHE_ARCHIVE.exists() or len(list(CACHE.glob('*.png'))) != len(work) + len(msk):
    tmp_archive = CACHE_ARCHIVE.with_suffix('.zip.tmp')
    with zipfile.ZipFile(tmp_archive, 'w', compression=zipfile.ZIP_STORED) as archive:
        for path in CACHE.glob('*.png'):
            archive.write(path, path.name)
    os.replace(tmp_archive, CACHE_ARCHIVE)
    print(f'Cache archivé sur Drive : {CACHE_ARCHIVE}')


In [ ]:
# ── 4. Données et encodeur clinique C0 ───────────────────────────────────
import yaml, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from src.data.datasets import build_transforms, ClinicalEncoder
from src.data.splits import assert_no_patient_leakage

cfg = yaml.safe_load(open('configs/data.yaml'))
cfg['preprocessing']['image_size'] = 224
tf_train, tf_val = build_transforms(cfg, True), build_transforms(cfg, False)

class CachedDS(Dataset):
    """Lit le cache 256 px ; renvoie image, clinique, label et masque."""
    def __init__(self, df, transform, enc=None, masks=False):
        self.df, self.tf, self.enc, self.masks = df.reset_index(drop=True), transform, enc, masks
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = cv2.cvtColor(cv2.imread(r.cached, cv2.IMREAD_GRAYSCALE), cv2.COLOR_GRAY2RGB)
        m = None
        if self.masks and isinstance(r.get('cached_mask'), str):
            mm = cv2.imread(r.cached_mask, cv2.IMREAD_GRAYSCALE)
            m = (mm > 127).astype(np.uint8) if mm is not None else None
        out = self.tf(image=img, mask=m) if m is not None else self.tf(image=img)
        s = {'image': out['image'], 'label': torch.tensor(float(r.label)), 'index': i}
        if self.enc is not None:
            s['clinical'] = torch.from_numpy(self.enc.transform_row(r))
        if self.masks:
            s['mask'] = out['mask'].float() if m is not None else torch.zeros(224, 224)
        return s

echo = work[work.dataset == 'busi']
mammo = work[work.dataset == 'cmmd']

def split(df):
    tr = df[(df.fold >= 0) & (df.fold != 4)]
    va = df[df.fold == 4]
    test = df[df.fold == -1]
    assert_no_patient_leakage({'train': tr, 'val': va, 'test': test})
    return tuple(part.reset_index(drop=True) for part in (tr, va, test))

echo_tr, echo_val, echo_test = split(echo)
mam_tr, mam_val, mam_test = split(mammo)

# Encodeur clinique ajusté sur le TRAIN mammographie uniquement (C0 réel :
# âge de CMMD + type de lésion). Ajuster sur tout le jeu ferait fuiter le test.
ENC = ClinicalEncoder(continuous=['age'], binary=[], categorical=['lesion_type'])
ENC.fit(mam_tr)
print(f'Variables cliniques C0 : age (réel), lesion_type → {ENC.n_features} entrées')
print(f'échographie : train {len(echo_tr)} / val {len(echo_val)} / test {len(echo_test)}')
print(f'mammographie: train {len(mam_tr)} / val {len(mam_val)} / test {len(mam_test)} · âge présent {mam_tr.age.notna().mean():.0%}')

In [ ]:
# ── 5. Baselines mono-modaux ─────────────────────────────────────────────
from src.models.encoders import MonoModalClassifier, ClinicalOnlyClassifier
from src.train.trainer import Trainer, TrainConfig
from src.eval.metrics import compute_metrics, aggregate_patient_scores, best_threshold

ARCH, EPOCHS = 'resnet18', 15
BASELINE_IMAGE_BATCH_SIZE, CLINICAL_BATCH_SIZE, FUSION_BATCH_SIZE = 32, 64, 16
R, P, VALIDATION, VP, RUN_FINGERPRINTS = {}, {}, {}, {}, {}
TEST_PAIR_SEED, VAL_PAIR_SEED = 9001, 8001

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

protocol_files = [
    REPO / 'src/train/trainer.py', REPO / 'src/models/fusion.py',
    REPO / 'src/models/encoders.py', REPO / 'src/data/datasets.py',
    REPO / 'src/data/splits.py', REPO / 'src/eval/metrics.py',
    REPO / 'src/eval/delong.py', REPO / 'src/explain/gradcam.py',
    REPO / 'src/explain/localization.py', REPO / 'src/explain/shap_clinical.py',
    REPO / 'configs/data.yaml', REPO / 'data/manifest.csv',
]
protocol_payload = {str(path.relative_to(REPO)): sha256_file(path) for path in protocol_files}
notebook_sources = [cell.get('source', []) for cell in json.loads(
    (REPO / 'notebooks/04_resultats_presentation.ipynb').read_text(encoding='utf-8')
)['cells']]
protocol_payload['notebook_sources'] = hashlib.sha256(
    json.dumps(notebook_sources, ensure_ascii=False).encode()
).hexdigest()
PROTOCOL_HASH = hashlib.sha256(json.dumps(protocol_payload, sort_keys=True).encode()).hexdigest()

def run_fingerprint(name, **spec):
    payload = {'protocol_hash': PROTOCOL_HASH, 'run_name': name, **spec}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()

def atomic_json(path, payload):
    path = Path(path); tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(tmp, path)

def write_csv_atomic(frame, path, index=True):
    path = Path(path); tmp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(tmp, index=index); os.replace(tmp, path)

def save_progress():
    if R:
        write_csv_atomic(pd.DataFrame(R).T, OUT / 'metriques_test_provisoires.csv')
    if VALIDATION:
        write_csv_atomic(pd.DataFrame(VALIDATION).T, OUT / 'metriques_validation.csv')
        import matplotlib.pyplot as plt
        scores = {name: values['auc_roc'] for name, values in VALIDATION.items()}
        fig, ax = plt.subplots(figsize=(max(8, len(scores) * .45), 5))
        ax.bar(scores.keys(), scores.values()); ax.set_ylabel('AUC patient — validation')
        ax.set_title('Progression validation — ne pas interpréter comme test final')
        ax.tick_params(axis='x', labelrotation=75); fig.tight_layout()
        fig.savefig(FIG_DIR / 'progress_validation_auc.png', dpi=140); plt.close(fig)

def load_completed_run(name, fingerprint):
    path = RUN_DIR / f'{name}.json'
    if not path.exists(): return False
    record = json.loads(path.read_text(encoding='utf-8'))
    if record.get('status') != 'complete' or record.get('fingerprint') != fingerprint:
        raise ValueError(f'Résultat existant incompatible avec le protocole : {path}')
    checkpoint = torch.load(CKPT / f'{name}_best.pt', map_location='cpu', weights_only=True)
    if checkpoint.get('fingerprint') != fingerprint or 'model_state' not in checkpoint:
        raise ValueError(f'Checkpoint incompatible avec le résultat : {name}')
    fit_record = record.get('fit', {})
    if (checkpoint.get('epoch') != fit_record.get('best_epoch')
            or round(float(checkpoint.get('score', np.nan)), 4) != fit_record.get('best_score')):
        raise ValueError(f'Checkpoint et historique d’entraînement désalignés : {name}')
    state_tensors = list(checkpoint['model_state'].values())
    if (not state_tensors or any(not torch.is_tensor(value) for value in state_tensors)
            or any(torch.is_floating_point(value) and not torch.isfinite(value).all().item()
                   for value in state_tensors)):
        raise ValueError(f'Poids du checkpoint vides ou non finis : {name}')
    vy, vs = np.asarray(record['validation']['labels']), np.asarray(record['validation']['scores'])
    ty, ts = np.asarray(record['test']['labels']), np.asarray(record['test']['scores'])
    vi, ti = np.asarray(record['validation']['patient_ids']), np.asarray(record['test']['patient_ids'])
    if not (len(vy) == len(vs) == len(vi) and len(ty) == len(ts) == len(ti)):
        raise ValueError(f'Prédictions incomplètes ou désalignées : {name}')
    if (len(np.unique(vi)) != len(vi) or len(np.unique(ti)) != len(ti)
            or not np.isin(vy, [0, 1]).all() or not np.isin(ty, [0, 1]).all()
            or not np.isfinite(vs).all() or not np.isfinite(ts).all()):
        raise ValueError(f'Prédictions patientes invalides : {name}')
    threshold = float(record['threshold'])
    if not np.isclose(threshold, best_threshold(vy, vs)):
        raise ValueError(f'Seuil de décision non dérivé des prédictions de validation : {name}')
    if record.get('status') != 'complete':
        raise ValueError(f'Run marqué incomplet : {name}')
    val_metrics = compute_metrics(vy, vs, threshold=threshold)
    test_metrics = compute_metrics(ty, ts, threshold=threshold)
    if val_metrics != record['validation']['metrics'] or test_metrics != record['test']['metrics']:
        raise ValueError(f'Métriques persistées incohérentes avec les prédictions : {name}')
    VALIDATION[name], R[name] = val_metrics, test_metrics
    VP[name] = (vs, vy, vi)
    P[name] = (ts, ty, ti)
    RUN_FINGERPRINTS[name] = fingerprint
    return True

def persist_run(name, fingerprint, val_y, val_scores, val_ids, test_y, test_scores,
                test_ids, threshold, fit_result, extra=None):
    record = {
        'status': 'complete', 'run_name': name, 'fingerprint': fingerprint,
        'threshold': float(threshold), 'fit': fit_result,
        'validation': {'labels': np.asarray(val_y, dtype=int).tolist(),
                       'scores': np.asarray(val_scores, dtype=float).tolist(),
                       'patient_ids': [str(x) for x in val_ids],
                       'metrics': VALIDATION[name]},
        'test': {'labels': np.asarray(test_y, dtype=int).tolist(),
                 'scores': np.asarray(test_scores, dtype=float).tolist(),
                 'patient_ids': [str(x) for x in test_ids], 'metrics': R[name]},
    }
    if extra: record.update(extra)
    atomic_json(RUN_DIR / f'{name}.json', record)
    VP[name] = (np.asarray(val_scores, dtype=float), np.asarray(val_y, dtype=int), np.asarray(val_ids))
    RUN_FINGERPRINTS[name] = fingerprint
    save_progress()

def save_training_plot(name):
    history_path = CKPT / f'{name}_history.json'
    if not history_path.exists(): return
    history = json.loads(history_path.read_text(encoding='utf-8'))['history']
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot([row['epoch'] for row in history], [row['auc_roc'] for row in history], marker='.')
    ax.set(xlabel='Époque', ylabel='AUC patient — validation', title=f'Historique : {name}')
    ax.grid(alpha=.25); fig.tight_layout(); fig.savefig(FIG_DIR / f'{name}_history.png', dpi=140)
    plt.close(fig)

def run(name, model, tr_ds, val_ds, test_ds, bs=32, fwd=None, dataset_name=None):
    spec = {'kind': 'unimodal', 'dataset': dataset_name, 'architecture': ARCH,
            'epochs': EPOCHS, 'batch_size': bs, 'seed': 42,
            'train_patients': tr_ds.df.patient_id.astype(str).tolist(),
            'validation_patients': val_ds.df.patient_id.astype(str).tolist(),
            'test_patients': test_ds.df.patient_id.astype(str).tolist()}
    fingerprint = run_fingerprint(name, **spec)
    if load_completed_run(name, fingerprint):
        state = torch.load(CKPT / f'{name}_best.pt', map_location='cpu', weights_only=False)
        model.load_state_dict(state['model_state']); model.cpu(); save_progress()
        (CKPT / f'{name}_last.pt').unlink(missing_ok=True)
        save_training_plot(name)
        return model
    tl = DataLoader(tr_ds, batch_size=bs, shuffle=True, num_workers=2, drop_last=False)
    vl = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=2)
    tr = Trainer(model, tl, vl, TrainConfig(epochs=EPOCHS, batch_size=bs, patience=5, run_name=name,
                                            out_dir=CKPT, seed=42), forward_fn=fwd,
                  validation_groups=val_ds.df.patient_id.to_numpy())
    fit_result = tr.fit(resume=True, fingerprint=fingerprint); tr.load_best()
    val_sc, val_tg = tr.predict(vl)
    val_y, val_sc, val_ids = aggregate_patient_scores(val_tg, val_sc, val_ds.df.patient_id)
    threshold = best_threshold(val_y, val_sc)
    VALIDATION[name] = compute_metrics(val_y, val_sc, threshold=threshold)
    sc, tg = tr.predict(test_loader)
    patient_y, patient_sc, patient_ids = aggregate_patient_scores(tg, sc, test_ds.df.patient_id)
    R[name] = compute_metrics(patient_y, patient_sc, threshold=threshold)
    P[name] = (patient_sc, patient_y, patient_ids)
    persist_run(name, fingerprint, val_y, val_sc, val_ids, patient_y, patient_sc,
                patient_ids, threshold, fit_result)
    if not load_completed_run(name, fingerprint):
        raise RuntimeError(f'La vérification après sauvegarde a échoué : {name}')
    save_training_plot(name)
    print(f"  → {name}: AUC patient test {R[name]['auc_roc']:.4f}")
    model_cpu = tr.model.cpu()
    del tr, tl, vl, test_loader
    (CKPT / f'{name}_last.pt').unlink(missing_ok=True)
    gc.collect(); torch.cuda.empty_cache()
    return model_cpu

print('\n[1/3] Échographie seule (BUSI)')
torch.manual_seed(42)
t_echo = run('echo_seule', MonoModalClassifier(ARCH, True, 224),
             CachedDS(echo_tr, tf_train), CachedDS(echo_val, tf_val),
             CachedDS(echo_test, tf_val), bs=BASELINE_IMAGE_BATCH_SIZE, dataset_name='busi')

print('\n[2/3] Mammographie seule (CMMD)')
torch.manual_seed(42)
t_mam = run('mammo_seule', MonoModalClassifier(ARCH, True, 224),
            CachedDS(mam_tr, tf_train), CachedDS(mam_val, tf_val),
            CachedDS(mam_test, tf_val), bs=BASELINE_IMAGE_BATCH_SIZE, dataset_name='cmmd')

# La clinique seule a besoin d'une passe avant dédiée (pas d'image) : cellule 5b.


In [ ]:
# ── 5b. Clinique seule (même reprise et journalisation) ─────────────────
torch.manual_seed(42)
clinical_model = ClinicalOnlyClassifier(ENC.n_features)
clin_model = run(
    'clinique_seule', clinical_model,
    CachedDS(mam_tr, tf_val, ENC), CachedDS(mam_val, tf_val, ENC),
    CachedDS(mam_test, tf_val, ENC), bs=CLINICAL_BATCH_SIZE,
    fwd=lambda b: clinical_model(b['clinical'].cuda()), dataset_name='cmmd',
)
auc_c = R['clinique_seule']['auc_roc']
print(f"AUC clinique seule (hold-out, seuil validation) = {auc_c:.4f}")
print('Contrôle descriptif uniquement : le type de lésion est réel mais peut être associé au diagnostic.')


In [ ]:
# ── 6. Fusion multi-modale sur appariement VIRTUEL ───────────────────────
# ⚠️ Les paires associent des patientes DIFFÉRENTES, sans consulter les labels :
#    aucun jeu public n'apparie mammographie et échographie chez la même femme.
#    Toute comparaison reste exploratoire et ne démontre pas une fusion
#    clinique chez une patiente réelle ; logit averaging est la référence.
import gc
from src.models.fusion import FUSION_REGISTRY, build_fusion

class PairDS(Dataset):
    """Paires virtuelles tirées sans consulter les labels d'aucun split."""
    def __init__(self, mdf, edf, tf, enc, seed=0):
        self.m = mdf.reset_index(drop=True); self.e = edf.reset_index(drop=True)
        if self.m.empty or self.e.empty:
            raise ValueError('Impossible de créer des paires avec un split vide.')
        self.tf, self.enc = tf, enc
        rng = np.random.default_rng(seed)
        ei = rng.choice(len(self.e), size=len(self.m), replace=len(self.e) < len(self.m))
        self.pairs = list(zip(range(len(self.m)), ei.tolist()))
    def __len__(self): return len(self.pairs)
    def _img(self, df, i):
        g = cv2.imread(df.iloc[i].cached, cv2.IMREAD_GRAYSCALE)
        if g is None: raise FileNotFoundError(f"Image de cache illisible : {df.iloc[i].cached}")
        return self.tf(image=cv2.cvtColor(g, cv2.COLOR_GRAY2RGB))['image']
    def __getitem__(self, k):
        mi, ei = self.pairs[k]; r = self.m.iloc[mi]
        return {'mammo': self._img(self.m, mi), 'us': self._img(self.e, ei),
                'clinical': torch.from_numpy(self.enc.transform_row(r)),
                'label': torch.tensor(float(r.label))}

FUSION_WEIGHTS, AMW_PATIENT_WEIGHTS = {}, {}

def run_fusion(strategy, seed=0, epochs=12, bs=FUSION_BATCH_SIZE):
    name = f'{strategy}_pair{seed}'
    tr_ds = PairDS(mam_tr, echo_tr, tf_train, ENC, seed)
    val_ds = PairDS(mam_val, echo_val, tf_val, ENC, VAL_PAIR_SEED)
    test_ds = PairDS(mam_test, echo_test, tf_val, ENC, TEST_PAIR_SEED)
    fingerprint = run_fingerprint(name, kind='fusion', strategy=strategy,
        architecture=ARCH, epochs=epochs, batch_size=bs, training_pair_seed=seed,
        validation_pair_seed=VAL_PAIR_SEED, test_pair_seed=TEST_PAIR_SEED,
        training_pairs=tr_ds.pairs, validation_pairs=val_ds.pairs, test_pairs=test_ds.pairs)
    if load_completed_run(name, fingerprint):
        record = json.loads((RUN_DIR / f'{name}.json').read_text(encoding='utf-8'))
        if 'modality_weights' in record: FUSION_WEIGHTS[name] = record['modality_weights']
        if 'patient_modality_weights' in record: AMW_PATIENT_WEIGHTS[name] = record['patient_modality_weights']
        save_progress()
        (CKPT / f'{name}_last.pt').unlink(missing_ok=True)
        save_training_plot(name)
        return
    t0 = time.time()
    try:
        torch.manual_seed(42)
        model = build_fusion(strategy, encoder_name=ARCH, pretrained=True,
                             clinical_dim=ENC.n_features, image_size=224).cuda()
        tl = DataLoader(tr_ds, batch_size=bs, shuffle=True, num_workers=2, drop_last=False)
        vl = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=2)
        test_loader = DataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=2)
        fwd = lambda b: model(b['mammo'].cuda(), b['us'].cuda(), b['clinical'].cuda())
        tr = Trainer(model, tl, vl, TrainConfig(epochs=epochs, batch_size=bs, patience=4, run_name=name,
                                                out_dir=CKPT, seed=42), forward_fn=fwd,
                       validation_groups=val_ds.m.patient_id.to_numpy())
        fit_result = tr.fit(resume=True, fingerprint=fingerprint); tr.load_best()
        val_sc, val_tg = tr.predict(vl)
        val_y, val_sc, val_ids = aggregate_patient_scores(val_tg, val_sc, val_ds.m.patient_id)
        threshold = best_threshold(val_y, val_sc)
        VALIDATION[name] = compute_metrics(val_y, val_sc, threshold=threshold)
        sc, tg = tr.predict(test_loader)
        patient_y, patient_sc, patient_ids = aggregate_patient_scores(tg, sc, test_ds.m.patient_id)
        R[name] = compute_metrics(patient_y, patient_sc, threshold=threshold)
        P[name] = (patient_sc, patient_y, patient_ids)
        weights = model.modality_weights() if strategy == 'logit_averaging' else None
        extra = {'strategy': strategy, 'training_pair_seed': seed}
        if weights is not None:
            FUSION_WEIGHTS[name] = weights; extra['modality_weights'] = weights
        if strategy == 'cross_attention_amw':
            model.eval(); image_weights = []
            with torch.no_grad():
                for batch in test_loader:
                    model(batch['mammo'].cuda(), batch['us'].cuda(), batch['clinical'].cuda())
                    image_weights.extend(model.last_modality_weights.cpu().numpy())
            image_weights = np.asarray(image_weights, dtype=float)
            if image_weights.shape != (len(test_ds), 3) or not np.isfinite(image_weights).all():
                raise ValueError(f'Poids AMW invalides pour {name} : {image_weights.shape}')
            weight_frame = pd.DataFrame(image_weights, columns=['mammo','us','clinical'])
            weight_frame['patient_id'] = test_ds.m.patient_id.astype(str).to_numpy()
            patient_weights = weight_frame.groupby('patient_id').mean().reindex(patient_ids)
            if patient_weights.isna().any().any() or not np.allclose(patient_weights.sum(axis=1), 1.0, atol=1e-5):
                raise ValueError(f'Poids AMW patients incomplets ou non normalisés pour {name}.')
            AMW_PATIENT_WEIGHTS[name] = {'patient_ids':[str(x) for x in patient_ids],
                'names':['mammo','us','clinical'], 'weights':patient_weights.to_numpy().tolist()}
            extra['patient_modality_weights'] = AMW_PATIENT_WEIGHTS[name]
        persist_run(name, fingerprint, val_y, val_sc, val_ids, patient_y,
                    patient_sc, patient_ids, threshold, fit_result, extra)
        if not load_completed_run(name, fingerprint):
            raise RuntimeError(f'La vérification après sauvegarde a échoué : {name}')
        save_training_plot(name)
        print(f"  → {name}: AUC patient test {R[name]['auc_roc']:.4f} ({(time.time()-t0)/60:.1f} min)")
        model.cpu()
        del tr, model, tl, vl, test_loader
        (CKPT / f'{name}_last.pt').unlink(missing_ok=True)
    finally:
        torch.cuda.empty_cache(); gc.collect()

print('Cinq stratégies × cinq tirages d’appariement indépendants des labels :')
PRIORITY_ORDER = ['logit_averaging', 'late', 'cross_attention',
                  'cross_attention_amw', 'early']
if set(PRIORITY_ORDER) != set(FUSION_REGISTRY):
    raise ValueError('La liste de priorité doit couvrir chaque stratégie de fusion exactement une fois.')
RUN_EARLY_FUSION = True  # False = exécuter le noyau prioritaire, documenter early comme non exécuté.
STRATEGIES_TO_RUN = PRIORITY_ORDER if RUN_EARLY_FUSION else PRIORITY_ORDER[:-1]
for strategy in STRATEGIES_TO_RUN:
    for pair_seed in range(5):
        run_fusion(strategy, seed=pair_seed)

best_strategy = max(
    STRATEGIES_TO_RUN,
    key=lambda s: np.mean([VALIDATION[f'{s}_pair{i}']['auc_roc'] for i in range(5)])
)
print('\nStratégie retenue sur validation :', best_strategy)
print('Test d’intégrité : mêmes modèles, échographies du hold-out permutées sans réentraînement')
best_runs = [f'{best_strategy}_pair{i}' for i in range(5)]
val_y, val_ids = VP[best_runs[0]][1], VP[best_runs[0]][2]
for name in best_runs[1:]:
    if not np.array_equal(val_ids, VP[name][2]) or not np.array_equal(val_y, VP[name][1]):
        raise ValueError(f'Validation désalignée entre tirages de {best_strategy}.')
val_ensemble = np.mean(np.stack([VP[name][0] for name in best_runs]), axis=0)
control_threshold = best_threshold(val_y, val_ensemble)
val_ensemble_metrics = compute_metrics(val_y, val_ensemble, threshold=control_threshold)
VALIDATION[f'{best_strategy}_ensemble'] = val_ensemble_metrics
TEST_IDS = P[best_runs[0]][2]
test_y = P[best_runs[0]][1]
for name in best_runs[1:]:
    if not np.array_equal(TEST_IDS, P[name][2]) or not np.array_equal(test_y, P[name][1]):
        raise ValueError(f'Test désaligné entre tirages de {best_strategy}.')
original_ensemble = np.mean(np.stack([P[name][0] for name in best_runs]), axis=0)
control_name = 'integrity_shuffle'
control_fingerprint = run_fingerprint(control_name, selected_strategy=best_strategy,
    model_fingerprints=[RUN_FINGERPRINTS[name] for name in best_runs],
    test_pair_seed=TEST_PAIR_SEED, permutation_seed=1907)
control_path = RUN_DIR / f'{control_name}.json'
if control_path.exists():
    control = json.loads(control_path.read_text(encoding='utf-8'))
    if control.get('fingerprint') != control_fingerprint or control.get('threshold') != control_threshold:
        raise ValueError('Contrôle existant incompatible avec les modèles ou le protocole.')
    shuffled_y = np.asarray(control['shuffled']['labels'], dtype=int)
    shuffled_scores = np.asarray(control['shuffled']['scores'], dtype=float)
    shuffled_ids = np.asarray(control['shuffled']['patient_ids'])
    original_saved = np.asarray(control['original']['scores'], dtype=float)
    if (not np.array_equal(np.asarray(control['original']['labels'], dtype=int), test_y)
            or not np.array_equal(np.asarray(control['original']['patient_ids']), TEST_IDS)):
        raise ValueError('Cohorte ou labels originaux du contrôle incohérents.')
    if len(np.unique(shuffled_ids)) != len(shuffled_ids) or not np.isfinite(shuffled_scores).all():
        raise ValueError('Prédictions patientes invalides dans le contrôle persisté.')
    if compute_metrics(shuffled_y, shuffled_scores, threshold=control_threshold) != control['shuffled']['metrics']:
        raise ValueError('Métriques persistées du contrôle d’intégrité incohérentes.')
    original_metrics = compute_metrics(test_y, original_ensemble, threshold=control_threshold)
    if control.get('status') != 'complete' or not np.array_equal(original_saved, original_ensemble):
        raise ValueError('Prédictions originales du contrôle incohérentes avec les cinq modèles.')
    if original_metrics != control['original']['metrics']:
        raise ValueError('Métriques originales du contrôle incohérentes avec les prédictions.')
else:
    permuted_test = PairDS(mam_test, echo_test, tf_val, ENC, TEST_PAIR_SEED)
    rng = np.random.default_rng(1907)
    shuffled_us = rng.permutation(len(permuted_test.e))
    original_us_indices = np.asarray([ui for _, ui in permuted_test.pairs])
    permuted_us_indices = shuffled_us[original_us_indices]
    if np.array_equal(permuted_us_indices, original_us_indices):
        raise ValueError('La permutation n’a modifié aucune paire effective du hold-out.')
    permuted_test.pairs = [(mi, int(shuffled_us[ui])) for mi, ui in permuted_test.pairs]
    perm_loader = DataLoader(permuted_test, batch_size=FUSION_BATCH_SIZE, shuffle=False, num_workers=2)
    per_model_scores = []
    for name in best_runs:
        state = torch.load(CKPT / f'{name}_best.pt', map_location='cpu', weights_only=False)
        if state.get('fingerprint') != RUN_FINGERPRINTS[name]:
            raise ValueError(f'Checkpoint invalide pour le contrôle : {name}')
        model = build_fusion(best_strategy, encoder_name=ARCH, pretrained=False,
                             clinical_dim=ENC.n_features, image_size=224).cuda()
        model.load_state_dict(state['model_state']); model.eval()
        scores = []
        with torch.no_grad():
            for batch in perm_loader:
                logits = model(batch['mammo'].cuda(), batch['us'].cuda(), batch['clinical'].cuda())
                scores.extend(torch.sigmoid(logits).cpu().numpy().tolist())
        per_model_scores.append(np.asarray(scores))
        del model; torch.cuda.empty_cache()
    shuffled_image_scores = np.mean(np.stack(per_model_scores), axis=0)
    shuffled_y, shuffled_scores, shuffled_ids = aggregate_patient_scores(
        mam_test.label.to_numpy(), shuffled_image_scores, mam_test.patient_id.to_numpy())
    if not np.array_equal(shuffled_ids, TEST_IDS) or not np.array_equal(shuffled_y, test_y):
        raise ValueError('Cohorte ou labels du contrôle d’intégrité désalignés.')
    shuffled_metrics = compute_metrics(shuffled_y, shuffled_scores, threshold=control_threshold)
    original_metrics = compute_metrics(test_y, original_ensemble, threshold=control_threshold)
    control = {'status':'complete', 'fingerprint':control_fingerprint,
        'selected_strategy':best_strategy, 'threshold':control_threshold,
        'original':{'labels':test_y.tolist(), 'scores':original_ensemble.tolist(),
                    'patient_ids':[str(x) for x in TEST_IDS], 'metrics':original_metrics},
        'shuffled':{'labels':shuffled_y.tolist(), 'scores':shuffled_scores.tolist(),
                    'patient_ids':[str(x) for x in shuffled_ids], 'metrics':shuffled_metrics}}
    atomic_json(control_path, control)
if not np.array_equal(shuffled_ids, TEST_IDS) or not np.array_equal(shuffled_y, test_y):
    raise ValueError('Cohorte ou labels du contrôle d’intégrité différents du hold-out.')
R[control_name] = control['shuffled']['metrics']
P[control_name] = (shuffled_scores, shuffled_y, shuffled_ids)
RUN_FINGERPRINTS[control_name] = control_fingerprint
save_progress()
print(f"AUC moyenne originale : {compute_metrics(test_y, original_ensemble, threshold=control_threshold)['auc_roc']:.4f}")
print(f"AUC après permutation de l’échographie : {R[control_name]['auc_roc']:.4f}")

print('\n' + '='*66)
print(pd.DataFrame(R).T[['auc_roc']].sort_values('auc_roc', ascending=False).round(4).to_string())


In [ ]:
# ── 7. TABLEAU PRINCIPAL DE LA SOUTENANCE ────────────────────────────────
STRATEGIES = list(STRATEGIES_TO_RUN)
for strategy in STRATEGIES:
    runs = [f'{strategy}_pair{i}' for i in range(5)]
    ids, targets = P[runs[0]][2], P[runs[0]][1]
    val_ids, val_targets = VP[runs[0]][2], VP[runs[0]][1]
    for name in runs[1:]:
        if not np.array_equal(ids, P[name][2]) or not np.array_equal(targets, P[name][1]):
            raise ValueError(f'Patientes ou labels test désalignés entre tirages de {strategy}.')
        if not np.array_equal(val_ids, VP[name][2]) or not np.array_equal(val_targets, VP[name][1]):
            raise ValueError(f'Patientes ou labels validation désalignés entre tirages de {strategy}.')
    mean_scores = np.mean(np.stack([P[name][0] for name in runs]), axis=0)
    mean_val_scores = np.mean(np.stack([VP[name][0] for name in runs]), axis=0)
    threshold = best_threshold(val_targets, mean_val_scores)
    VALIDATION[strategy] = compute_metrics(val_targets, mean_val_scores, threshold=threshold)
    R[strategy] = compute_metrics(targets, mean_scores, threshold=threshold)
    R[strategy]['auc_roc_mean_seed'] = float(np.mean([R[name]['auc_roc'] for name in runs]))
    R[strategy]['auc_roc_std'] = float(np.std([R[name]['auc_roc'] for name in runs], ddof=1))
    P[strategy] = (mean_scores, targets, ids)
for baseline in ('echo_seule', 'mammo_seule', 'clinique_seule'):
    R[baseline]['auc_roc_mean_seed'] = R[baseline]['auc_roc']
    R[baseline]['auc_roc_std'] = np.nan

rows = {k: R[k] for k in ['echo_seule', 'mammo_seule', 'clinique_seule'] + STRATEGIES}
tbl = pd.DataFrame(rows).T[['auc_roc','auc_roc_mean_seed','auc_roc_std','sensitivity','specificity','f1','mcc','spec_at_sens95']]
tbl.columns = ['AUC ensemble/test','AUC moyenne seeds','ET AUC seeds','Sensibilité','Spécificité','F1','MCC','Spéc.@Sens95']
tbl = tbl.round(4)
print('Les lignes BUSI et CMMD portent sur des cohortes différentes : ne pas les classer entre elles.')
print('Fusion : scores moyennés sur cinq appariements pour les métriques/DeLong ; AUC moyenne ± ET des graines reportées séparément.')
print('Baselines : entraînement unique ; AUC moyenne seeds = AUC du run, écart-type non disponible.')
display(tbl)
write_csv_atomic(tbl, OUT / 'tableau_principal.csv')

In [ ]:
# ── 8. Test de DeLong — LA cellule qui décide de vos conclusions ─────────
# Un écart d'AUC ne vaut rien sans test statistique. DeLong compare deux AUC
# sur les MÊMES patientes, en tenant compte de leur corrélation.
from src.eval.delong import delong_test, auc_ci

FUSIONS = list(STRATEGIES)

print('Intervalles de confiance à 95 % :')
for k in FUSIONS:
    ci = auc_ci(P[k][1], P[k][0])
    print(f"  {k + '_ensemble':<32} AUC {ci['auc']:.4f}  [{ci['ci_low']:.3f} ; {ci['ci_high']:.3f}]")
for k in ['mammo_seule','echo_seule','clinique_seule']:
    ci = auc_ci(P[k][1], P[k][0])
    print(f"  {k:<32} AUC {ci['auc']:.4f}  [{ci['ci_low']:.3f} ; {ci['ci_high']:.3f}]")

def cmp(a, b):
    if a not in P or b not in P: return None
    if not np.array_equal(P[a][2], P[b][2]):
        raise ValueError(f"Comparaison invalide : cohortes patientes différentes ({a}, {b}).")
    if not np.array_equal(P[a][1], P[b][1]):
        raise ValueError(f"Comparaison invalide : labels patients différents ({a}, {b}).")
    return delong_test(P[a][1], P[a][0], P[b][0], a, b)

rows = []
# ⭐ chaque stratégie contre la référence sans interaction
for s in FUSIONS:
    if s != 'logit_averaging':
        rows.append(cmp(s, 'logit_averaging'))
rows.append(cmp('cross_attention_amw', 'cross_attention'))
# Le test fusion/mammographie porte sur les mêmes patientes CMMD.
best_fusion = best_strategy if FUSIONS else None
if best_fusion:
    rows.append(cmp(best_fusion, 'mammo_seule'))
    rows.append(cmp(best_fusion, 'clinique_seule'))
# Les prédictions restent alignées sur les mêmes patientes et labels.
rows.append(cmp(best_strategy, 'integrity_shuffle'))

dl = pd.DataFrame([r for r in rows if r])
order = np.argsort(dl.p_value.to_numpy())
adjusted = np.empty(len(dl), dtype=float)
running = 0.0
for rank, index in enumerate(order):
    running = max(running, (len(order) - rank) * float(dl.iloc[index].p_value))
    adjusted[index] = min(1.0, running)
dl['p_holm'] = adjusted
dl['significant_holm'] = dl.p_holm < 0.05
display(dl[['name_a','name_b','auc_a','auc_b','delta','p_value','p_holm','significant_holm']])
write_csv_atomic(dl, OUT / 'delong.csv', index=False)

print('\n' + '='*66)
print('⭐ VERDICT — les mécanismes d’attention apportent-ils quelque chose ?')
print('='*66)
for _, r in dl.iterrows():
    if r['name_b'] == 'logit_averaging':
        verdict = 'différence après correction de Holm' if r['significant_holm'] else 'pas de différence après correction de Holm'
        print(f"  {r['name_a']:<24} Δ={r['delta']:+.4f}  p_Holm={r['p_holm']:.4f}  → {verdict}")

if best_fusion and not dl[dl.name_b=='logit_averaging'].significant_holm.any():
    print('\n→ Aucune stratégie ne bat la moyenne des logits après correction de Holm.')
    print('  Sur données non appariées, l’attention n’apporte rien de plus qu’un')
    print('  vote pondéré. Résultat honnête, publiable, et rarement vérifié')
    print('  dans la littérature — présentez-le comme votre contribution.')


In [ ]:
# ── 9. Grad-CAM validé par IoU (l’apport méthodologique du projet) ───────
# L’état de l’art MONTRE des cartes de chaleur ; nous les MESURONS.
from src.explain.gradcam import GradCAM, overlay_cam
from src.explain.localization import evaluate_localization, localization_report

eval_df = echo_test[echo_test.cached_mask.notna()].reset_index(drop=True)
ds = CachedDS(eval_df, tf_val, masks=True)
dl_ = DataLoader(ds, batch_size=16, shuffle=False, num_workers=2)

model = t_echo.model.eval().cuda()
cams, masks_l, labels_l, preds_l = [], [], [], []
cam_gen = GradCAM(model, model.image_encoder.backbone)
for b in dl_:
    x = b['image'].cuda().requires_grad_(True)
    c = cam_gen(x)
    with torch.no_grad():
        pr = (torch.sigmoid(model(b['image'].cuda())) > 0.5).int().cpu().numpy()
    cams += list(c); masks_l += list(b['mask'].numpy())
    labels_l += list(b['label'].numpy().astype(int)); preds_l += list(pr)
cam_gen.remove()

loc = evaluate_localization(cams, masks_l, np.array(labels_l), np.array(preds_l))
rep = localization_report(loc)
print(json.dumps(rep, indent=2, ensure_ascii=False))
write_csv_atomic(loc, OUT / 'localisation.csv', index=False)
atomic_json(OUT / 'localisation_resume.json', rep)

print(f"\n⭐ CHIFFRE POUR LA SOUTENANCE : le modèle vise la lésion dans "
      f"{rep['pointing_game']:.0%} des cas (IoU moyen {rep['iou_mean']:.3f}).")

In [ ]:
# ── 10. Figures pour les diapositives ────────────────────────────────────
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# (a) Courbes ROC CMMD — patientes et labels identiques
plt.figure(figsize=(6.5, 5.5))
for k in FUSIONS + ['mammo_seule', 'clinique_seule']:
    if k in P:
        f, t, _ = roc_curve(P[k][1], P[k][0])
        curve_auc = roc_auc_score(P[k][1], P[k][0])
        plt.plot(f, t, lw=2, label=f"{k} — AUC {curve_auc:.3f}")
plt.plot([0,1],[0,1],'k--',alpha=.3); plt.xlabel('1 − spécificité'); plt.ylabel('sensibilité')
plt.title('CMMD — fusion vs mammographie / clinique'); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT/'fig_roc.png', dpi=150); plt.show()

# BUSI est une cohorte distincte : sa baseline est tracée séparément.
f, t, _ = roc_curve(P['echo_seule'][1], P['echo_seule'][0])
plt.figure(figsize=(5.5, 4.5)); plt.plot(f, t, label=f"Échographie BUSI — AUC {roc_auc_score(P['echo_seule'][1], P['echo_seule'][0]):.3f}")
plt.plot([0,1],[0,1],'k--',alpha=.3); plt.xlabel('1 − spécificité'); plt.ylabel('sensibilité')
plt.title('BUSI — échographie seule'); plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(OUT/'fig_roc_busi.png', dpi=150); plt.show()

# (b) Grad-CAM : visés / manqués
hit = loc[loc.pointing == 1].head(4).index.tolist()
miss = loc[loc.pointing == 0].head(4).index.tolist()
sel = [(i,'visé') for i in hit] + [(i,'manqué') for i in miss]
if sel:
    fig, axes = plt.subplots(2, len(sel)//2 or 1, figsize=(3*(len(sel)//2 or 1), 6.5))
    for ax, (i, tag) in zip(np.ravel(axes), sel):
        row = loc.iloc[i]; j = int(row['index'])
        img = cv2.imread(eval_df.iloc[j].cached, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (224,224))
        ax.imshow(overlay_cam(img, cams[j])); ax.axis('off')
        ax.set_title(f"{tag} · IoU {row['iou']:.2f}", fontsize=9)
    plt.suptitle('Grad-CAM — le modèle regarde-t-il la lésion ?')
    plt.tight_layout(); plt.savefig(OUT/'fig_gradcam.png', dpi=150); plt.show()

# (c) Point de fonctionnement clinique
plt.figure(figsize=(6.5,4))
ks = FUSIONS + ['mammo_seule']
x = np.arange(len(ks)); w = 0.4
plt.bar(x-w/2, [R[k]['auc_roc'] for k in ks], w, label='AUC')
plt.bar(x+w/2, [R[k]['spec_at_sens95'] for k in ks], w, label='Spécificité à 95 % de sensibilité')
plt.xticks(x, ks, rotation=20, ha='right'); plt.legend(fontsize=8); plt.grid(alpha=.3, axis='y')
plt.title('AUC ≠ utilité clinique — cohorte CMMD'); plt.tight_layout()
plt.savefig(OUT/'fig_clinique.png', dpi=150); plt.show()


In [ ]:
# ── 11. SHAP sur les variables cliniques réelles ─────────────────────────
from src.explain.shap_clinical import clinical_shap_values, shap_summary_table, plot_shap_summary

names = [f'age_{s}' for s in ('valeur','present')] + \
        [f'lesion_{c}' for c in ENC.categories_.get('lesion_type', [])]
X = np.stack([ENC.transform_row(mam_test.iloc[i]) for i in range(min(120, len(mam_test)))])
bg = np.stack([ENC.transform_row(mam_tr.iloc[i]) for i in range(min(80, len(mam_tr)))])

vals = clinical_shap_values(clin_model.cpu(), bg, X, names[:ENC.n_features], nsamples=100)
tblshap = shap_summary_table(vals, names[:ENC.n_features])
display(tblshap); write_csv_atomic(tblshap, OUT/'shap.csv', index=False)
plot_shap_summary(vals, names[:ENC.n_features]).savefig(OUT/'fig_shap.png', dpi=150)
plt.show()

amw_runs = [f'cross_attention_amw_pair{i}' for i in range(5)]
amw_arrays = []
for name in amw_runs:
    saved = AMW_PATIENT_WEIGHTS.get(name)
    if saved is None or saved['names'] != ['mammo','us','clinical']:
        raise ValueError(f'Poids AMW persistés absents ou invalides : {name}')
    ids = np.asarray(saved['patient_ids'])
    if not np.array_equal(ids, P[name][2]):
        raise ValueError(f'Identifiants des poids AMW désalignés : {name}')
    arr = np.asarray(saved['weights'], dtype=float)
    if arr.shape != (len(ids), 3) or not np.isfinite(arr).all() or not np.allclose(arr.sum(axis=1), 1.0, atol=1e-5):
        raise ValueError(f'Poids AMW patients invalides : {name}')
    amw_arrays.append(arr)
amw_stack = np.stack(amw_arrays)
amw_mean = amw_stack.mean(axis=0); amw_std = amw_stack.std(axis=0, ddof=1)
amw_table = pd.DataFrame({'patient_id':P[amw_runs[0]][2]})
for i, modality in enumerate(['mammo','us','clinical']):
    amw_table[f'{modality}_mean'] = amw_mean[:, i]
    amw_table[f'{modality}_std_pairing'] = amw_std[:, i]
write_csv_atomic(amw_table, OUT/'amw_patient_weights.csv', index=False)
print('Poids AMW moyens sur le hold-out (descriptifs, cinq tirages) :',
      {m: float(amw_mean[:, i].mean()) for i, m in enumerate(['mammo','us','clinical'])})

# Poids de modalité appris — quelle modalité domine ?
logit_runs = [f'logit_averaging_pair{i}' for i in range(5)]
if all(name in FUSION_WEIGHTS for name in logit_runs):
    keys = FUSION_WEIGHTS[logit_runs[0]].keys()
    if any(FUSION_WEIGHTS[name].keys() != FUSION_WEIGHTS[logit_runs[0]].keys() for name in logit_runs[1:]):
        raise ValueError('Les poids de modalités ne sont pas comparables entre tirages.')
    mean_weights = {key: float(np.mean([FUSION_WEIGHTS[name][key] for name in logit_runs])) for key in keys}
    print('\nPoids de modalités appris — moyenne des cinq appariements :', mean_weights)
else:
    raise ValueError('Poids du modèle logit_averaging manquants dans les résultats validés.')


In [ ]:
# ── 12. Synthèse à recopier dans les diapositives ────────────────────────
best = best_strategy
lines = [
    '='*68, 'RÉSULTATS — SOUTENANCE', '='*68,
    f"Données : {len(echo)} échographies (BUSI) + {len(mammo)} mammographies (CMMD)",
    f"          {mammo.patient_id.nunique()} patientes avec âge réel · variables C0 uniquement", '',
    'PERFORMANCES', tbl.to_string(), '',
    f"Fusion retenue sur validation : {best} (AUC ensemble hold-out {R[best]['auc_roc']:.4f})",
    f"Baseline clinique descriptive : AUC CMMD {R['clinique_seule']['auc_roc']:.3f} "
    f"({'seuil d’alerte ≤ 0,65' if R['clinique_seule']['auc_roc'] <= 0.65 else 'à examiner > 0,65'})", '',
    "  Le type lesionnel est une variable réelle, mais son association au diagnostic impose une interprétation prudente.", '',
    'EXPLICABILITÉ MESURÉE (et non illustrée)',
    f"  Le modèle vise la lésion dans {rep['pointing_game']:.0%} des cas",
    f"  IoU moyen carte/masque : {rep['iou_mean']:.3f} (sur {rep['n_images']} images)",
    f"  {rep.get('interpretation','')}", '',
    'LIMITE ASSUMÉE',
    "  Aucun jeu public n'apparie mammographie et échographie chez la même",
    "  patiente : les paires sont virtuelles et tirées sans consulter le diagnostic.",
    "  D'où la comparaison systématique à la moyenne des logits.",
    "  L'échographie BUSI est une cohorte distincte et n'est pas classée avec CMMD.", '='*68,
]
txt = '\n'.join(lines); print(txt)
(OUT/'synthese.txt').write_text(txt, encoding='utf-8')
print(f'\n✅ Tout est dans {OUT} : métriques, DeLong, localisation, 5 figures, synthèse.')

## Comment présenter ces résultats

**Ne menez pas avec l'AUC.** Menez avec la question clinique : *« à 95 % de sensibilité, combien de biopsies inutiles évite-t-on ? »* — c'est la figure `fig_clinique.png`.

**Assumez l'appariement virtuel dès la première diapositive de méthode.** Un jury qui le découvre en questions le perçoit comme une faiblesse cachée ; annoncé d'emblée et mesuré, c'est une rigueur méthodologique. La formulation : *« aucun jeu public n'apparie les deux modalités chez la même patiente ; nous construisons donc des paires virtuelles conditionnées au diagnostic, et comparons systématiquement à une combinaison sans interaction »*.

**Si la cross-attention ne bat pas la moyenne des logits** (issue la plus probable) : ce n'est pas un échec, c'est votre résultat. *« Sur données non appariées, le mécanisme d'attention n'apporte rien de plus qu'un vote pondéré — ce que la littérature ne vérifie généralement pas. »*

**Votre différenciateur : le Grad-CAM mesuré.** Latha et Chen montrent des cartes de chaleur ; vous annoncez un pourcentage. C'est la diapositive qui vous distingue.

**Trois questions probables du jury**

| Question | Réponse |
|---|---|
| « Vos images viennent-elles des mêmes patientes ? » | Non, et c'est documenté — voir la méthode et le test d'intégrité |
| « Pourquoi si peu de variables cliniques ? » | Seules les variables réellement mesurées sont utilisées ; en inventer produirait une fuite de cible |
| « 0,93 d'AUC, est-ce utilisable ? » | À 98 % de sensibilité, la spécificité chute — c'est la marge de progression, et nous la chiffrons |